# Phase 1 - Contrôle Qualité global des datasets

Ce notebook applique une logique de contrôle qualité commune à l'ensemble des fichiers du projet.

## Objectifs

Ce notebook permet de :

- charger tous les principaux datasets
- obtenir un aperçu global de chaque table
- détecter les valeurs manquantes
- détecter les doublons
- vérifier l'unicité des identifiants principaux lorsque cela est pertinent
- produire un résumé synthétique de qualité pour chaque fichier

## Remarque

Ce notebook ne remplace pas les analyses détaillées par dataset.  
Il sert de vue d'ensemble rapide pour accélérer le projet et identifier les fichiers qui nécessitent une attention particulière.

## 1. Import des bibliothèques

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

## 2. Chargement des fichiers

In [2]:
PROJECT_ROOT = Path.cwd().parent
DATA_RAW = PROJECT_ROOT / "data" / "raw"

dataset_paths = {
    "application_train": DATA_RAW / "application_train.csv",
    "application_test": DATA_RAW / "application_test.csv",
    "bureau": DATA_RAW / "bureau.csv",
    "bureau_balance": DATA_RAW / "bureau_balance.csv",
    "previous_application": DATA_RAW / "previous_application.csv",
    "installments_payments": DATA_RAW / "installments_payments.csv",
    "credit_card_balance": DATA_RAW / "credit_card_balance.csv",
    "POS_CASH_balance": DATA_RAW / "POS_CASH_balance.csv",
}

datasets = {name: pd.read_csv(path) for name, path in dataset_paths.items()}

pd.DataFrame({
    "dataset": list(datasets.keys()),
    "shape": [df.shape for df in datasets.values()]
})

,dataset,shape
0,application_train,"(307511, 122)"
1,application_test,"(48744, 121)"
2,bureau,"(1716428, 17)"
3,bureau_balance,"(27299925, 3)"
4,previous_application,"(1670214, 37)"
5,installments_payments,"(13605401, 8)"
6,credit_card_balance,"(3840312, 23)"
7,POS_CASH_balance,"(10001358, 8)"


## 3. Fonctions utilitaires de contrôle qualité

In [3]:
primary_keys = {
    "application_train": "SK_ID_CURR",
    "application_test": "SK_ID_CURR",
    "bureau": "SK_ID_BUREAU",
    "previous_application": "SK_ID_PREV",
}

def build_overview(df: pd.DataFrame) -> pd.DataFrame:
    overview = pd.DataFrame({
        "colonne": df.columns,
        "type": df.dtypes.astype(str).values,
        "nb_non_null": df.notna().sum().values,
        "nb_manquant": df.isna().sum().values,
        "pct_manquant": (df.isna().mean().values * 100),
        "n_unique": df.nunique(dropna=True).values,
    })
    return overview.sort_values(by="pct_manquant", ascending=False)

def build_dataset_summary(name: str, df: pd.DataFrame) -> dict:
    pk = primary_keys.get(name)
    duplicate_pk = None

    if pk is not None and pk in df.columns:
        duplicate_pk = int(df[pk].duplicated().sum())

    return {
        "dataset": name,
        "n_lignes": df.shape[0],
        "n_colonnes": df.shape[1],
        "nb_colonnes_manquantes": int((df.isna().sum() > 0).sum()),
        "nb_colonnes_>50pct_nan": int((df.isna().mean() > 0.50).sum()),
        "nb_doublons_lignes": int(df.duplicated().sum()),
        "id_principal": pk if pk is not None else "Non défini",
        "doublons_id_principal": duplicate_pk if duplicate_pk is not None else "N/A"
    }

## 4. Résumé global de qualité

In [4]:
global_summary = pd.DataFrame(
    [build_dataset_summary(name, df) for name, df in datasets.items()]
)

global_summary

,dataset,n_lignes,n_colonnes,nb_colonnes_manquantes,nb_colonnes_>50pct_nan,nb_doublons_lignes,id_principal,doublons_id_principal
0,application_train,307511,122,67,41,0,SK_ID_CURR,0
1,application_test,48744,121,64,29,0,SK_ID_CURR,0
2,bureau,1716428,17,7,2,0,SK_ID_BUREAU,0
3,bureau_balance,27299925,3,0,0,0,Non défini,N/A
4,previous_application,1670214,37,16,4,0,SK_ID_PREV,0
5,installments_payments,13605401,8,2,0,0,Non défini,N/A
6,credit_card_balance,3840312,23,9,0,0,Non défini,N/A
7,POS_CASH_balance,10001358,8,2,0,0,Non défini,N/A


### Analyse globale des datasets

L'analyse montre que le projet repose sur plusieurs tables avec des niveaux de granularité différents.

- `application_train` et `application_test` sont les tables principales au niveau client
- les autres datasets contiennent des informations historiques ou transactionnelles
- certaines tables présentent un nombre élevé de lignes, ce qui indique une granularité fine (ex : paiements, historiques)

Cette structure confirme que le pipeline devra intégrer une étape d’agrégation avant la modélisation.

## 5. Aperçu détaillé par dataset

In [5]:
for name, df in datasets.items():
    print("=" * 100)
    print(f"Dataset : {name}")
    print(f"Dimensions : {df.shape}")
    display(df.head(3))

Dataset : application_train
Dimensions : (307511, 122)


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_TYPE_SUITE,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,DAYS_ID_PUBLISH,OWN_CAR_AGE,FLAG_MOBIL,FLAG_EMP_PHONE,FLAG_WORK_PHONE,FLAG_CONT_MOBILE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,REGION_RATING_CLIENT,REGION_RATING_CLIENT_W_CITY,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,REG_REGION_NOT_LIVE_REGION,REG_REGION_NOT_WORK_REGION,LIVE_REGION_NOT_WORK_REGION,REG_CITY_NOT_LIVE_CITY,REG_CITY_NOT_WORK_CITY,LIVE_CITY_NOT_WORK_CITY,ORGANIZATION_TYPE,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3,APARTMENTS_AVG,BASEMENTAREA_AVG,YEARS_BEGINEXPLUATATION_AVG,YEARS_BUILD_AVG,COMMONAREA_AVG,ELEVATORS_AVG,ENTRANCES_AVG,FLOORSMAX_AVG,FLOORSMIN_AVG,LANDAREA_AVG,LIVINGAPARTMENTS_AVG,LIVINGAREA_AVG,NONLIVINGAPARTMENTS_AVG,NONLIVINGAREA_AVG,APARTMENTS_MODE,BASEMENTAREA_MODE,YEARS_BEGINEXPLUATATION_MODE,YEARS_BUILD_MODE,COMMONAREA_MODE,ELEVATORS_MODE,ENTRANCES_MODE,FLOORSMAX_MODE,FLOORSMIN_MODE,LANDAREA_MODE,LIVINGAPARTMENTS_MODE,LIVINGAREA_MODE,NONLIVINGAPARTMENTS_MODE,NONLIVINGAREA_MODE,APARTMENTS_MEDI,BASEMENTAREA_MEDI,YEARS_BEGINEXPLUATATION_MEDI,YEARS_BUILD_MEDI,COMMONAREA_MEDI,ELEVATORS_MEDI,ENTRANCES_MEDI,FLOORSMAX_MEDI,FLOORSMIN_MEDI,LANDAREA_MEDI,LIVINGAPARTMENTS_MEDI,LIVINGAREA_MEDI,NONLIVINGAPARTMENTS_MEDI,NONLIVINGAREA_MEDI,FONDKAPREMONT_MODE,HOUSETYPE_MODE,TOTALAREA_MODE,WALLSMATERIAL_MODE,EMERGENCYSTATE_MODE,OBS_30_CNT_SOCIAL_CIRCLE,DEF_30_CNT_SOCIAL_CIRCLE,OBS_60_CNT_SOCIAL_CIRCLE,DEF_60_CNT_SOCIAL_CIRCLE,DAYS_LAST_PHONE_CHANGE,FLAG_DOCUMENT_2,FLAG_DOCUMENT_3,FLAG_DOCUMENT_4,FLAG_DOCUMENT_5,FLAG_DOCUMENT_6,FLAG_DOCUMENT_7,FLAG_DOCUMENT_8,FLAG_DOCUMENT_9,FLAG_DOCUMENT_10,FLAG_DOCUMENT_11,FLAG_DOCUMENT_12,FLAG_DOCUMENT_13,FLAG_DOCUMENT_14,FLAG_DOCUMENT_15,FLAG_DOCUMENT_16,FLAG_DOCUMENT_17,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,"202,500.0000","406,597.5000","24,700.5000","351,000.0000",Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.0188,-9461,-637,"-3,648.0000",-2120,NaN,1,1,0,1,1,0,Laborers,1.0000,2,2,WEDNESDAY,10,0,0,0,0,0,0,Business Entity Type 3,0.0830,0.2629,0.1394,0.0247,0.0369,0.9722,0.6192,0.0143,0.0000,0.0690,0.0833,0.1250,0.0369,0.0202,0.0190,0.0000,0.0000,0.0252,0.0383,0.9722,0.6341,0.0144,0.0000,0.0690,0.0833,0.1250,0.0377,0.0220,0.0198,0.0000,0.0000,0.0250,0.0369,0.9722,0.6243,0.0144,0.0000,0.0690,0.0833,0.1250,0.0375,0.0205,0.0193,0.0000,0.0000,reg oper account,block of flats,0.0149,"Stone, brick",No,2.0000,2.0000,2.0000,2.0000,"-1,134.0000",0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000
1,100003,0,Cash loans,F,N,N,0,"270,000.0000","1,293,502.5000","35,698.5000","1,129,500.0000",Family,State servant,Higher education,Married,House / apartment,0.0035,-16765,-1188,"-1,186.0000",-291,NaN,1,1,0,1,1,0,Core staff,2.0000,1,1,MONDAY,11,0,0,0,0,0,0,School,0.3113,0.6222,NaN,0.0959,0.0529,0.9851,0.7960,0.0605,0.0800,0.0345,0.2917,0.3333,0.0130,0.0773,0.0549,0.0039,0.0098,0.0924,0.0538,0.9851,0.8040,0.0497,0.0806,0.0345,0.2917,0.3333,0.0128,0.0790,0.0554,0.0000,0.0000,0.0968,0.0529,0.9851,0.7987,0.0608,0.0800,0.0345,0.2917,0.3333,0.0132,0.0787,0.0558,0.0039,0.0100,reg oper account,block of flats,0.0714,Block,No,1.0000,0.0000,1.0000,0.0000,-828.0000,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
2,100004,0,Revolving loans,M,Y,Y,0,"67,500.0000","135,000.0000","6,750.0000","135,000.0000",Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.0100,-19046,-225,"-4,260.0000",-2531,26.0000,1,1,1,1,1,0,Laborers,1.0000

Dataset : application_test
Dimensions : (48744, 121)


,SK_ID_CURR,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_TYPE_SUITE,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,DAYS_ID_PUBLISH,OWN_CAR_AGE,FLAG_MOBIL,FLAG_EMP_PHONE,FLAG_WORK_PHONE,FLAG_CONT_MOBILE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,REGION_RATING_CLIENT,REGION_RATING_CLIENT_W_CITY,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,REG_REGION_NOT_LIVE_REGION,REG_REGION_NOT_WORK_REGION,LIVE_REGION_NOT_WORK_REGION,REG_CITY_NOT_LIVE_CITY,REG_CITY_NOT_WORK_CITY,LIVE_CITY_NOT_WORK_CITY,ORGANIZATION_TYPE,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3,APARTMENTS_AVG,BASEMENTAREA_AVG,YEARS_BEGINEXPLUATATION_AVG,YEARS_BUILD_AVG,COMMONAREA_AVG,ELEVATORS_AVG,ENTRANCES_AVG,FLOORSMAX_AVG,FLOORSMIN_AVG,LANDAREA_AVG,LIVINGAPARTMENTS_AVG,LIVINGAREA_AVG,NONLIVINGAPARTMENTS_AVG,NONLIVINGAREA_AVG,APARTMENTS_MODE,BASEMENTAREA_MODE,YEARS_BEGINEXPLUATATION_MODE,YEARS_BUILD_MODE,COMMONAREA_MODE,ELEVATORS_MODE,ENTRANCES_MODE,FLOORSMAX_MODE,FLOORSMIN_MODE,LANDAREA_MODE,LIVINGAPARTMENTS_MODE,LIVINGAREA_MODE,NONLIVINGAPARTMENTS_MODE,NONLIVINGAREA_MODE,APARTMENTS_MEDI,BASEMENTAREA_MEDI,YEARS_BEGINEXPLUATATION_MEDI,YEARS_BUILD_MEDI,COMMONAREA_MEDI,ELEVATORS_MEDI,ENTRANCES_MEDI,FLOORSMAX_MEDI,FLOORSMIN_MEDI,LANDAREA_MEDI,LIVINGAPARTMENTS_MEDI,LIVINGAREA_MEDI,NONLIVINGAPARTMENTS_MEDI,NONLIVINGAREA_MEDI,FONDKAPREMONT_MODE,HOUSETYPE_MODE,TOTALAREA_MODE,WALLSMATERIAL_MODE,EMERGENCYSTATE_MODE,OBS_30_CNT_SOCIAL_CIRCLE,DEF_30_CNT_SOCIAL_CIRCLE,OBS_60_CNT_SOCIAL_CIRCLE,DEF_60_CNT_SOCIAL_CIRCLE,DAYS_LAST_PHONE_CHANGE,FLAG_DOCUMENT_2,FLAG_DOCUMENT_3,FLAG_DOCUMENT_4,FLAG_DOCUMENT_5,FLAG_DOCUMENT_6,FLAG_DOCUMENT_7,FLAG_DOCUMENT_8,FLAG_DOCUMENT_9,FLAG_DOCUMENT_10,FLAG_DOCUMENT_11,FLAG_DOCUMENT_12,FLAG_DOCUMENT_13,FLAG_DOCUMENT_14,FLAG_DOCUMENT_15,FLAG_DOCUMENT_16,FLAG_DOCUMENT_17,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100001,Cash loans,F,N,Y,0,"135,000.0000","568,800.0000","20,560.5000","450,000.0000",Unaccompanied,Working,Higher education,Married,House / apartment,0.0188,-19241,-2329,"-5,170.0000",-812,NaN,1,1,0,1,0,1,NaN,2.0000,2,2,TUESDAY,18,0,0,0,0,0,0,Kindergarten,0.7526,0.7897,0.1595,0.0660,0.0590,0.9732,NaN,NaN,NaN,0.1379,0.1250,NaN,NaN,NaN,0.0505,NaN,NaN,0.0672,0.0612,0.9732,NaN,NaN,NaN,0.1379,0.1250,NaN,NaN,NaN,0.0526,NaN,NaN,0.0666,0.0590,0.9732,NaN,NaN,NaN,0.1379,0.1250,NaN,NaN,NaN,0.0514,NaN,NaN,NaN,block of flats,0.0392,"Stone, brick",No,0.0000,0.0000,0.0000,0.0000,"-1,740.0000",0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
1,100005,Cash loans,M,N,Y,0,"99,000.0000","222,768.0000","17,370.0000","180,000.0000",Unaccompanied,Working,Secondary / secondary special,Married,House / apartment,0.0358,-18064,-4469,"-9,118.0000",-1623,NaN,1,1,0,1,0,0,Low-skill Laborers,2.0000,2,2,FRIDAY,9,0,0,0,0,0,0,Self-employed,0.5650,0.2917,0.4330,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0000,0.0000,0.0000,0.0000,0.0000,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0000,0.0000,0.0000,0.0000,0.0000,3.0000
2,100013,Cash loans,M,Y,Y,0,"202,500.0000","663,264.0000","69,777.0000","630,000.0000",NaN,Working,Higher education,Married,House / apartment,0.0191,-20038,-4458,"-2,175.0000",-3503,5.0000,1,1,0,1,0,0,Drivers,2.0000,2,2,MONDAY,14,0,0,0,0,0,0,Transport: type 3,NaN,0.6998,0.6110,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0000,0.0000,0.0000,0.0000,-856.0000,0,0,0,0,0,0,1,0,0,0,0

Dataset : bureau
Dimensions : (1716428, 17)


...

,SK_ID_PREV,SK_ID_CURR,NUM_INSTALMENT_VERSION,NUM_INSTALMENT_NUMBER,DAYS_INSTALMENT,DAYS_ENTRY_PAYMENT,AMT_INSTALMENT,AMT_PAYMENT
0,1054186,161674,1.0000,6,"-1,180.0000","-1,187.0000","6,948.3600","6,948.3600"
1,1330831,151639,0.0000,34,"-2,156.0000","-2,156.0000","1,716.5250","1,716.5250"
2,2085231,193053,2.0000,1,-63.0000,-63.0000,"25,425.0000","25,425.0000"


Dataset : credit_card_balance
Dimensions : (3840312, 23)


,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,AMT_BALANCE,AMT_CREDIT_LIMIT_ACTUAL,AMT_DRAWINGS_ATM_CURRENT,AMT_DRAWINGS_CURRENT,AMT_DRAWINGS_OTHER_CURRENT,AMT_DRAWINGS_POS_CURRENT,AMT_INST_MIN_REGULARITY,AMT_PAYMENT_CURRENT,AMT_PAYMENT_TOTAL_CURRENT,AMT_RECEIVABLE_PRINCIPAL,AMT_RECIVABLE,AMT_TOTAL_RECEIVABLE,CNT_DRAWINGS_ATM_CURRENT,CNT_DRAWINGS_CURRENT,CNT_DRAWINGS_OTHER_CURRENT,CNT_DRAWINGS_POS_CURRENT,CNT_INSTALMENT_MATURE_CUM,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
0,2562384,378907,-6,56.9700,135000,0.0000,877.5000,0.0000,877.5000,"1,700.3250","1,800.0000","1,800.0000",0.0000,0.0000,0.0000,0.0000,1,0.0000,1.0000,35.0000,Active,0,0
1,2582071,363914,-1,"63,975.5550",45000,"2,250.0000","2,250.0000",0.0000,0.0000,"2,250.0000","2,250.0000","2,250.0000","60,175.0800","64,875.5550","64,875.5550",1.0000,1,0.0000,0.0000,69.0000,Active,0,0
2,1740877,371185,-7,"31,815.2250",450000,0.0000,0.0000,0.0000,0.0000,"2,250.0000","2,250.0000","2,250.0000","26,926.4250","31,460.0850","31,460.0850",0.0000,0,0.0000,0.0000,30.0000,Active,0,0


Dataset : POS_CASH_balance
Dimensions : (10001358, 8)


,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,CNT_INSTALMENT,CNT_INSTALMENT_FUTURE,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
0,1803195,182943,-31,48.0000,45.0000,Active,0,0
1,1715348,367990,-33,36.0000,35.0000,Active,0,0
2,1784872,397406,-32,12.0000,9.0000,Active,0,0


## 6. Types de variables par dataset

In [6]:
dtype_reports = {}

for name, df in datasets.items():
    dtype_reports[name] = (
        df.dtypes.astype(str)
        .value_counts()
        .rename_axis("type")
        .reset_index(name="count")
    )

for name, report in dtype_reports.items():
    print("=" * 100)
    print(f"Types de variables - {name}")
    display(report)

Types de variables - application_train


,type,count
0,float64,65
1,int64,41
2,str,16


Types de variables - application_test


,type,count
0,float64,65
1,int64,40
2,str,16


Types de variables - bureau


...

,type,count
0,float64,5
1,int64,3


Types de variables - credit_card_balance


,type,count
0,float64,15
1,int64,7
2,str,1


Types de variables - POS_CASH_balance


,type,count
0,int64,5
1,float64,2
2,str,1


## 7. Valeurs manquantes par dataset 

In [7]:
for name, df in datasets.items():
    print("=" * 100)
    print(f"Valeurs manquantes - {name}")
    missing = (
        df.isna()
        .mean()
        .mul(100)
        .sort_values(ascending=False)
        .reset_index()
    )
    missing.columns = ["colonne", "pct_manquant"]
    missing["nb_manquant"] = missing["colonne"].map(df.isna().sum())
    display(missing.head(15))

Valeurs manquantes - application_train


,colonne,pct_manquant,nb_manquant
0,COMMONAREA_AVG,69.8723,214865
1,COMMONAREA_MODE,69.8723,214865
2,COMMONAREA_MEDI,69.8723,214865
3,NONLIVINGAPARTMENTS_MEDI,69.4330,213514
4,NONLIVINGAPARTMENTS_MODE,69.4330,213514
5,NONLIVINGAPARTMENTS_AVG,69.4330,213514
6,FONDKAPREMONT_MODE,68.3862,210295
7,LIVINGAPARTMENTS_AVG,68.3550,210199
8,LIVINGAPARTMENTS_MEDI,68.3550,210199
9,LIVINGAPARTMENTS_MODE,68.3550,210199


Valeurs manquantes - application_test


,colonne,pct_manquant,nb_manquant
0,COMMONAREA_AVG,68.7161,33495
1,COMMONAREA_MEDI,68.7161,33495
2,COMMONAREA_MODE,68.7161,33495
3,NONLIVINGAPARTMENTS_AVG,68.4125,33347
4,NONLIVINGAPARTMENTS_MEDI,68.4125,33347
5,NONLIVINGAPARTMENTS_MODE,68.4125,33347
6,FONDKAPREMONT_MODE,67.2842,32797
7,LIVINGAPARTMENTS_MEDI,67.2493,32780
8,LIVINGAPARTMENTS_AVG,67.2493,32780
9,LIVINGAPARTMENTS_MODE,67.2493,32780


Valeurs manquantes - bureau


...

,colonne,pct_manquant,nb_manquant
0,AMT_PAYMENT,0.0214,2905
1,DAYS_ENTRY_PAYMENT,0.0214,2905
2,SK_ID_PREV,0.0000,0
3,SK_ID_CURR,0.0000,0
4,NUM_INSTALMENT_NUMBER,0.0000,0
5,NUM_INSTALMENT_VERSION,0.0000,0
6,DAYS_INSTALMENT,0.0000,0
7,AMT_INSTALMENT,0.0000,0


Valeurs manquantes - credit_card_balance


,colonne,pct_manquant,nb_manquant
0,AMT_PAYMENT_CURRENT,19.9981,767988
1,CNT_DRAWINGS_POS_CURRENT,19.5249,749816
2,AMT_DRAWINGS_ATM_CURRENT,19.5249,749816
3,CNT_DRAWINGS_ATM_CURRENT,19.5249,749816
4,AMT_DRAWINGS_POS_CURRENT,19.5249,749816
5,AMT_DRAWINGS_OTHER_CURRENT,19.5249,749816
6,CNT_DRAWINGS_OTHER_CURRENT,19.5249,749816
7,CNT_INSTALMENT_MATURE_CUM,7.9482,305236
8,AMT_INST_MIN_REGULARITY,7.9482,305236
9,AMT_DRAWINGS_CURRENT,0.0000,0


Valeurs manquantes - POS_CASH_balance


,colonne,pct_manquant,nb_manquant
0,CNT_INSTALMENT_FUTURE,0.2608,26087
1,CNT_INSTALMENT,0.2607,26071
2,SK_ID_CURR,0.0000,0
3,SK_ID_PREV,0.0000,0
4,MONTHS_BALANCE,0.0000,0
5,NAME_CONTRACT_STATUS,0.0000,0
6,SK_DPD,0.0000,0
7,SK_DPD_DEF,0.0000,0


### Analyse des valeurs manquantes

On observe que plusieurs colonnes présentent un taux élevé de valeurs manquantes.

Cela ne correspond pas nécessairement à des erreurs, mais peut indiquer :
- des informations non disponibles pour tous les clients
- des variables spécifiques à certains types de crédits
- des données optionnelles

Ces variables devront être traitées lors de la phase de prétraitement (imputation ou suppression).

## 8. Valeurs abérantes outliers

In [8]:
def detect_outliers_iqr(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]

    return len(outliers), lower_bound, upper_bound


# Exemple sur quelques variables clés
columns_to_check = ["AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY"]

outliers_summary = []

for col in columns_to_check:
    n_outliers, low, high = detect_outliers_iqr(datasets["application_train"], col)
    outliers_summary.append({
        "colonne": col,
        "nb_outliers": n_outliers,
        "borne_min": low,
        "borne_max": high
    })

pd.DataFrame(outliers_summary)

,colonne,nb_outliers,borne_min,borne_max
0,AMT_INCOME_TOTAL,14035,"-22,500.0000","337,500.0000"
1,AMT_CREDIT,6562,"-537,975.0000","1,616,625.0000"
2,AMT_ANNUITY,7504,"-10,584.0000","61,704.0000"


### Interprétation des valeurs aberrantes

L’analyse des valeurs aberrantes basée sur la méthode IQR met en évidence un nombre significatif de valeurs extrêmes dans les variables financières.

Cependant, dans le contexte du crédit, ces valeurs correspondent souvent à des clients ayant :
- des revenus élevés
- des montants de crédit importants
- des capacités de remboursement supérieures à la moyenne

Ainsi, ces observations ne sont pas considérées comme des anomalies mais comme des cas légitimes reflétant la diversité des profils clients.

Par conséquent, aucune suppression des outliers n’a été effectuée à ce stade.

## 9. Doublons et identifiants

In [9]:
duplicate_checks = []

for name, df in datasets.items():
    row_duplicates = int(df.duplicated().sum())
    pk = primary_keys.get(name)

    if pk is not None and pk in df.columns:
        pk_duplicates = int(df[pk].duplicated().sum())
    else:
        pk_duplicates = "N/A"

    duplicate_checks.append({
        "dataset": name,
        "doublons_lignes": row_duplicates,
        "id_principal": pk if pk is not None else "Non défini",
        "doublons_id_principal": pk_duplicates,
    })

duplicate_checks_df = pd.DataFrame(duplicate_checks)
duplicate_checks_df

,dataset,doublons_lignes,id_principal,doublons_id_principal
0,application_train,0,SK_ID_CURR,0
1,application_test,0,SK_ID_CURR,0
2,bureau,0,SK_ID_BUREAU,0
3,bureau_balance,0,Non défini,N/A
4,previous_application,0,SK_ID_PREV,0
5,installments_payments,0,Non défini,N/A
6,credit_card_balance,0,Non défini,N/A
7,POS_CASH_balance,0,Non défini,N/A


### Remarque

L'unicité de l'identifiant dépend du niveau de granularité de la table :

- `SK_ID_CURR` est l'identifiant client
- `SK_ID_BUREAU` identifie un enregistrement bureau
- `SK_ID_PREV` identifie une demande précédente

Aucun doublon exact de lignes n’a été détecté dans les datasets.

Cependant, certaines tables contiennent plusieurs enregistrements pour un même identifiant, ce qui est normal car elles représentent des données transactionnelles ou historiques.

## 10. Analyse ciblée de la variable cible

In [10]:
target_distribution = (
    datasets["application_train"]["TARGET"]
    .value_counts(dropna=False)
    .rename_axis("TARGET")
    .reset_index(name="count")
)

target_distribution["pct"] = target_distribution["count"] / target_distribution["count"].sum() * 100
target_distribution

,TARGET,count,pct
0,0,282686,91.9271
1,1,24825,8.0729


### Interprétation

Le dataset principal `application_train` contient une variable cible déséquilibrée.  
Cette caractéristique devra être prise en compte dans la phase de modélisation et dans l'analyse des métriques.

## 11. Vérifications métier  par dataset

In [11]:
# Liste qui va stocker le résumé des règles métier vérifiées
business_rules_summary = []

# =========================================================
# 1. application_train
# =========================================================
# Cette table est la table principale du projet.
# On y vérifie des règles métier de base sur des colonnes importantes
# pour la modélisation.

app_train = datasets["application_train"]

# Règle 1 :
# Le revenu total du client ne doit pas être négatif.
# Une valeur négative serait incohérente d’un point de vue métier.
business_rules_summary.append({
    "dataset": "application_train",
    "règle": "AMT_INCOME_TOTAL >= 0",
    "nb_violations": int((app_train["AMT_INCOME_TOTAL"] < 0).sum())
})

# Règle 2 :
# Dans ce dataset, DAYS_BIRTH est codé en nombre de jours relatifs
# par rapport à la date de demande. Cette valeur doit donc être négative,
# car la naissance est toujours dans le passé.
business_rules_summary.append({
    "dataset": "application_train",
    "règle": "DAYS_BIRTH < 0",
    "nb_violations": int((app_train["DAYS_BIRTH"] >= 0).sum())
})

# =========================================================
# 2. bureau
# =========================================================
# Cette table contient les informations de crédits externes.
# AMT_CREDIT_SUM représente un montant de crédit ; il doit donc être
# positif ou nul lorsqu’il est renseigné.
# Les valeurs manquantes sont autorisées.

bureau = datasets["bureau"]

if "AMT_CREDIT_SUM" in bureau.columns:
    business_rules_summary.append({
        "dataset": "bureau",
        "règle": "AMT_CREDIT_SUM >= 0 ou NaN",
        "nb_violations": int(
            ((bureau["AMT_CREDIT_SUM"] < 0) & bureau["AMT_CREDIT_SUM"].notna()).sum()
        )
    })

# =========================================================
# 3. previous_application
# =========================================================
# Cette table contient les anciennes demandes de crédit.
# AMT_APPLICATION correspond au montant demandé.
# Ce montant ne doit pas être négatif lorsqu’il existe.

prev = datasets["previous_application"]

if "AMT_APPLICATION" in prev.columns:
    business_rules_summary.append({
        "dataset": "previous_application",
        "règle": "AMT_APPLICATION >= 0 ou NaN",
        "nb_violations": int(
            ((prev["AMT_APPLICATION"] < 0) & prev["AMT_APPLICATION"].notna()).sum()
        )
    })

# =========================================================
# 4. installments_payments
# =========================================================
# Cette table contient les paiements par échéance.
# AMT_PAYMENT représente un montant payé.
# Un paiement ne doit pas être négatif lorsqu’il est renseigné.

inst = datasets["installments_payments"]

if "AMT_PAYMENT" in inst.columns:
    business_rules_summary.append({
        "dataset": "installments_payments",
        "règle": "AMT_PAYMENT >= 0 ou NaN",
        "nb_violations": int(
            ((inst["AMT_PAYMENT"] < 0) & inst["AMT_PAYMENT"].notna()).sum()
        )
    })

# =========================================================
# 5. credit_card_balance
# =========================================================
# Cette table contient l’historique des cartes de crédit.
# ATTENTION :
# AMT_BALANCE peut être positif, nul ou négatif.
# Une valeur négative n’est pas forcément une erreur métier :
# elle peut correspondre à un trop-perçu, un crédit en faveur du client,
# ou un cas spécifique de gestion du solde.
#
# Donc ici, on ne vérifie PAS que AMT_BALANCE >= 0.
# À la place, on indique explicitement qu’aucune règle simple de signe
# n’est appliquée sur cette variable.

ccb = datasets["credit_card_balance"]

if "AMT_BALANCE" in ccb.columns:
    business_rules_summary.append({
        "dataset": "credit_card_balance",
        "règle": "AMT_BALANCE peut être négatif, nul ou positif",
        "nb_violations": "Non applicable"
    })

# =========================================================
# 6. POS_CASH_balance
# =========================================================
# Cette table contient des informations de remboursement.
# CNT_INSTALMENT représente un nombre d’échéances.
# Un nombre d’échéances ne doit pas être négatif lorsqu’il est renseigné.

pos = datasets["POS_CASH_balance"]

if "CNT_INSTALMENT" in pos.columns:
    business_rules_summary.append({
        "dataset": "POS_CASH_balance",
        "règle": "CNT_INSTALMENT >= 0 ou NaN",
        "nb_violations": int(
            ((pos["CNT_INSTALMENT"] < 0) & pos["CNT_INSTALMENT"].notna()).sum()
        )
    })

# =========================================================
# Résultat final
# =========================================================
# On convertit la liste des règles en DataFrame pour obtenir
# un tableau lisible dans le notebook.
pd.DataFrame(business_rules_summary)

,dataset,règle,nb_violations
0,application_train,AMT_INCOME_TOTAL >= 0,0
1,application_train,DAYS_BIRTH < 0,0
2,bureau,AMT_CREDIT_SUM >= 0 ou NaN,0
3,previous_application,AMT_APPLICATION >= 0 ou NaN,0
4,installments_payments,AMT_PAYMENT >= 0 ou NaN,0
5,credit_card_balance,"AMT_BALANCE peut être négatif, nul ou positif",Non applicable
6,POS_CASH_balance,CNT_INSTALMENT >= 0 ou NaN,0


## 12. Synthèse finale

Cette première passe de contrôle qualité permet d'obtenir une vue rapide sur tous les datasets du projet.

Principaux constats :

- `application_train` reste la table centrale du projet
- plusieurs tables présentent des valeurs manquantes importantes
- certaines tables ont une granularité transactionnelle normale, donc plusieurs lignes par client
- les vérifications métier devront être approfondies au cas par cas
- cette vue globale facilite la priorisation des validations Pandera et YAML sur les fichiers secondaires

##  Prochaines étapes

À partir de ce notebook, la suite recommandée est :

1. définir un schéma Pandera minimal pour chaque dataset secondaire
2. créer un fichier YAML par dataset
3. préparer les agrégations nécessaires au feature engineering
4. fusionner ensuite les tables validées avec `application_train`


## Validation des données avec Pandera

Après l’analyse exploratoire, les règles métier sont maintenant automatisées avec Pandera.

Cela permet de :
- vérifier les types de données
- valider les contraintes métier
- sécuriser le pipeline avant le feature engineering


In [12]:

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from configs.pandera_schemas import (
    validate_application_train,
    validate_bureau,
    validate_previous_application,
    validate_installments_payments,
    validate_credit_card_balance,
    validate_pos_cash_balance,
    validate_bureau_balance
)


ModuleNotFoundError: No module named 'configs.pandera_schemas'

In [ ]:

train = datasets["application_train"]
bureau = datasets["bureau"]
previous = datasets["previous_application"]
installments = datasets["installments_payments"]
credit_card = datasets["credit_card_balance"]
pos_cash = datasets["POS_CASH_balance"]
bureau_balance = datasets["bureau_balance"]

print("Validation application_train :", validate_application_train(train).shape)
print("Validation bureau :", validate_bureau(bureau).shape)
print("Validation previous_application :", validate_previous_application(previous).shape)
print("Validation installments :", validate_installments_payments(installments).shape)
print("Validation credit_card :", validate_credit_card_balance(credit_card).shape)
print("Validation POS_CASH :", validate_pos_cash_balance(pos_cash).shape)
print("Validation bureau_balance :", validate_bureau_balance(bureau_balance).shape)



### Interprétation

Les validations Pandera confirment que les datasets respectent les contraintes définies.

Cette étape permet de détecter automatiquement les incohérences et garantit la robustesse du pipeline.



## Configuration des règles de validation (YAML)

Les règles métier sont également définies dans des fichiers YAML.

Cela permet :
- une meilleure lisibilité
- une séparation entre code et configuration
- une maintenance plus simple


In [ ]:

import yaml

with open(PROJECT_ROOT / "configs" / "application_train_quality.yaml", "r") as f:
    yaml_config = yaml.safe_load(f)

yaml_config



### Interprétation

Les fichiers YAML permettent de formaliser les règles métier de manière déclarative.

Ils complètent les validations Pandera et améliorent la traçabilité des règles appliquées.



## Conclusion du contrôle qualité

La phase de contrôle qualité a permis de :

- analyser la structure des données
- identifier les valeurs manquantes
- détecter les valeurs aberrantes
- vérifier les règles métier
- valider les données avec Pandera
- formaliser les règles avec YAML

Les données sont globalement cohérentes et prêtes pour la suite du projet.

La prochaine étape consiste à réaliser le feature engineering et la modélisation.
